# Portable LSFM Spot Viewer

JPEG化したRaw/前処理画像と、XYZだけに圧縮したTrackMate spotを表示します。`viewer`フォルダごと別PCへコピーして使用できます。

In [1]:
from functools import lru_cache
from io import BytesIO
from pathlib import Path
import zipfile

import ipywidgets as widgets
from IPython.display import display
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image


def find_viewer_root():
    candidates = [
        Path.cwd(),
        Path.cwd() / 'viewer',
        Path.cwd().parent / 'viewer',
        Path.cwd().parent,
    ]
    for candidate in candidates:
        if (candidate / 'manifest.csv').is_file():
            return candidate.resolve()
    raise FileNotFoundError('manifest.csv が見つかりません。viewerフォルダ内またはその隣でNotebookを開いてください。')


VIEWER_ROOT = find_viewer_root()
MANIFEST = pd.read_csv(VIEWER_ROOT / 'manifest.csv')
ROWS = MANIFEST.set_index('dataset').to_dict(orient='index')
print('Viewer root:', VIEWER_ROOT)
print('Datasets:', len(MANIFEST))


Viewer root: E:\LSFM_pp\viewer
Datasets: 40


In [2]:
@lru_cache(maxsize=8)
def load_spots(dataset):
    path = VIEWER_ROOT / ROWS[dataset]['spots_bundle']
    with np.load(path) as data:
        return {
            'x10': data['x10'].copy(),
            'y10': data['y10'].copy(),
            'offsets': data['offsets'].copy(),
            'before': int(data['spots_before_mask']),
            'after': int(data['spots_after_mask']),
            'mask_applied': bool(data['mask_applied']),
        }


@lru_cache(maxsize=24)
def load_image(dataset, source, z_index):
    column = 'raw_bundle' if source == 'Raw' else 'postprocessed_bundle'
    archive_path = VIEWER_ROOT / ROWS[dataset][column]
    with zipfile.ZipFile(archive_path, 'r') as archive:
        payload = archive.read('{:03d}.jpg'.format(z_index))
    return np.asarray(Image.open(BytesIO(payload)).convert('L'))


dataset_widget = widgets.Dropdown(
    options=MANIFEST['dataset'].tolist(), description='Dataset:',
    layout=widgets.Layout(width='390px')
)
source_widget = widgets.ToggleButtons(
    options=['Postprocessed', 'Raw'], value='Postprocessed', description='Image:'
)
z_widget = widgets.IntSlider(
    value=80, min=1, max=180, step=1, description='Z:',
    continuous_update=False, readout_format='d',
    layout=widgets.Layout(width='650px')
)
play_widget = widgets.Play(value=80, min=1, max=180, step=1, interval=250)
widgets.jslink((play_widget, 'value'), (z_widget, 'value'))
x_widget = widgets.IntSlider(
    value=2048, min=0, max=4095, step=16, description='X center:',
    continuous_update=False, layout=widgets.Layout(width='650px')
)
y_widget = widgets.IntSlider(
    value=1080, min=0, max=2159, step=8, description='Y center:',
    continuous_update=False, layout=widgets.Layout(width='650px')
)
zoom_widget = widgets.FloatLogSlider(
    value=1.0, base=2, min=-1, max=4, step=0.25, description='Zoom:',
    continuous_update=False, readout_format='.2f',
    layout=widgets.Layout(width='650px')
)
show_spots_widget = widgets.Checkbox(value=True, description='Show spots')
marker_widget = widgets.FloatSlider(
    value=12.0, min=2.0, max=40.0, step=1.0, description='Marker:',
    continuous_update=False, layout=widgets.Layout(width='430px')
)
range_widget = widgets.IntRangeSlider(
    value=(0, 255), min=0, max=255, step=1, description='Black/White:',
    continuous_update=False, readout=True,
    layout=widgets.Layout(width='650px')
)
gamma_widget = widgets.FloatLogSlider(
    value=1.0, base=2, min=-2, max=2, step=0.125, description='Gamma:',
    continuous_update=False, readout_format='.2f',
    layout=widgets.Layout(width='650px')
)
reset_intensity_widget = widgets.Button(description='Reset intensity')


def reset_intensity(_):
    range_widget.value = (0, 255)
    gamma_widget.value = 1.0


reset_intensity_widget.on_click(reset_intensity)


def reset_for_dataset(change):
    if change.get('name') != 'value':
        return
    row = ROWS[change['new']]
    x_widget.max = int(row['width']) - 1
    y_widget.max = int(row['height']) - 1
    z_widget.max = int(row['slices'])
    play_widget.max = int(row['slices'])
    x_widget.value = int(row['width']) // 2
    y_widget.value = int(row['height']) // 2


dataset_widget.observe(reset_for_dataset, names='value')


def render(dataset, source, z, x_center, y_center, zoom, intensity_range, gamma, show_spots, marker_size):
    row = ROWS[dataset]
    image = load_image(dataset, source, z - 1)
    height, width = image.shape
    effective_zoom = max(float(zoom), 1.0)
    viewport_width = width / effective_zoom
    viewport_height = height / effective_zoom
    x0 = np.clip(x_center - viewport_width / 2, 0, width - viewport_width)
    y0 = np.clip(y_center - viewport_height / 2, 0, height - viewport_height)
    x1 = x0 + viewport_width
    y1 = y0 + viewport_height

    display_scale = min(1.0, max(0.5, float(zoom)))
    black, white = intensity_range
    adjusted = np.clip(
        (image.astype(np.float32) - black) / max(float(white - black), 1.0),
        0.0, 1.0
    )
    adjusted = np.power(adjusted, 1.0 / max(float(gamma), 0.01))
    fig, axis = plt.subplots(figsize=(13 * display_scale, 7 * display_scale), dpi=110)
    axis.imshow(adjusted, cmap='gray', vmin=0, vmax=1, origin='upper',
                extent=(0, width, height, 0), interpolation='nearest')

    spots = load_spots(dataset)
    start, stop = spots['offsets'][z - 1:z + 1]
    count_in_slice = int(stop - start)
    count_visible = 0
    if show_spots and count_in_slice:
        xs = spots['x10'][start:stop].astype(np.float32) * 0.1
        ys = spots['y10'][start:stop].astype(np.float32) * 0.1
        visible = (xs >= x0) & (xs <= x1) & (ys >= y0) & (ys <= y1)
        count_visible = int(visible.sum())
        axis.scatter(xs[visible], ys[visible], s=marker_size,
                     facecolors='none', edgecolors='#39ff14', linewidths=0.65)

    axis.set_xlim(x0, x1)
    axis.set_ylim(y1, y0)
    mask_text = 'mask applied' if spots['mask_applied'] else 'MAY08: unmasked'
    axis.set_title(
        '{} | {} | Z={}/{} | zoom={:.2f}x | slice spots={:,} | visible={:,} | {}'.format(
            dataset, source, z, int(row['slices']), zoom,
            count_in_slice, count_visible, mask_text
        )
    )
    axis.set_xlabel('X [pixel]')
    axis.set_ylabel('Y [pixel]')
    plt.tight_layout()
    plt.show()


controls = widgets.VBox([
    widgets.HBox([dataset_widget, source_widget]),
    widgets.HBox([play_widget, z_widget]),
    x_widget,
    y_widget,
    zoom_widget,
    widgets.HBox([range_widget, reset_intensity_widget]),
    gamma_widget,
    widgets.HBox([show_spots_widget, marker_widget]),
])
output = widgets.interactive_output(
    render,
    {
        'dataset': dataset_widget,
        'source': source_widget,
        'z': z_widget,
        'x_center': x_widget,
        'y_center': y_widget,
        'zoom': zoom_widget,
        'intensity_range': range_widget,
        'gamma': gamma_widget,
        'show_spots': show_spots_widget,
        'marker_size': marker_widget,
    },
)
display(controls, output)


Output()